## Fuzzy matching engine code

In [16]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

# Load cleaned ingredient database
db_df = pd.read_csv("paulaschoice_us_ingredients_clean.csv")
db_df.head()




,Name,Description,Rating
0,Cranberry Seed Extract,Natural ingredient derived from antioxidant/om...,average
1,Crataegus Monogyna Extract,"Also known as hawthorn berry, this plant extra...",average
2,D&C,"According to the FDA, D&C is an identification...",average
3,D&C Colors,General term for a wide group of color additiv...,average
4,Daucus Carota,Also known as wild carrot. An antioxidant-rich...,average


In [17]:
db_names = db_df['Name'].str.lower().tolist()


In [18]:
# Example list of messy OCR ingredient strings
test_ocr_ingredients = [
    "ethylhexylglycerin",  # Typo example
    "Glycerin (Vegetable)",
    "Acetyl Hexapeptide-37",
    "Water/Aqua"
]

def clean_input(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s-]", "", text)  # Remove special chars except hyphen
    text = re.sub(r"\s+", " ", text).strip()
    return text

def match_ingredient(raw_ingredient):
    cleaned = clean_input(raw_ingredient)
    match, score, _ = process.extractOne(cleaned, db_names, scorer=fuzz.token_sort_ratio)
    return match, score

print("\n🧪 Ingredient Matching Results:")
for raw in test_ocr_ingredients:
    matched_name, confidence = match_ingredient(raw)
    print(f"OCR: {raw}  →  Match: {matched_name}  ({confidence}%)")

print("\n✅ Matching complete. Review and tune thresholds as needed.")


🧪 Ingredient Matching Results:
OCR: ethylhexylglycerin  →  Match: ethylhexylglycerin  (100.0%)
OCR: Glycerin (Vegetable)  →  Match: vegetable collagen  (72.22222222222221%)
OCR: Acetyl Hexapeptide-37  →  Match: acetyl hexapeptide-37  (100.0%)
OCR: Water/Aqua  →  Match: water  (71.42857142857143%)

✅ Matching complete. Review and tune thresholds as needed.


 ## Ingredient Matching Test & Manual Review Logger

In [22]:
import pandas as pd
from rapidfuzz import process, fuzz
import re

# Load cleaned ingredients database
db = pd.read_csv("paulaschoice_us_ingredients_clean.csv")

# Build master list: Names + Synonyms (for matching)
ingredient_list = db['Name'].tolist()




In [24]:
# Optional: If synonyms column exists, expand list
# Example:
if 'Synonyms' in db.columns:
     for syns in db['Synonyms'].dropna():
         ingredient_list.extend(eval(syns))  # Assuming synonyms stored as Python lists



In [25]:
# Function to clean and normalize input string
def clean_input(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Function to match ingredient
def match_ingredient(input_str, threshold=80):
    cleaned = clean_input(input_str)
    match, score, _ = process.extractOne(cleaned, ingredient_list, scorer=fuzz.token_sort_ratio)
    if score >= threshold:
        return match, score
    else:
        return None, score

# ----------- Test Matching -----------

unmatched = []

test_inputs = [
    "glycerine",  # Should match "Glycerin"
    "ethylhexyglycerin",  # Typo, should match "Ethylhexylglycerin"
    "sodium hualuronate",  # Should match "Sodium Hyaluronate"
    "xyzrandomchemical"  # Should be logged as unmatched
]

for test in test_inputs:
    match, score = match_ingredient(test)
    if match:
        print(f"✅ '{test}' matched to '{match}' with score {score}")
    else:
        print(f"❌ '{test}' no good match found (score {score}), logging.")
        unmatched.append(test)

# Log unmatched for manual review
if unmatched:
    pd.DataFrame(unmatched, columns=["Unmatched_Input"]).to_csv("unmatched_ingredients.csv", index=False)
    print(f"⚠️ Logged {len(unmatched)} unmatched ingredients to 'unmatched_ingredients.csv'")
else:
    print("🎉 All test inputs matched.")

✅ 'glycerine' matched to 'Glycerine' with score 88.88888888888889
✅ 'ethylhexyglycerin' matched to 'Ethylhexylglycerin' with score 91.42857142857143
✅ 'sodium hualuronate' matched to 'Sodium Hyaluronate' with score 83.33333333333334
❌ 'xyzrandomchemical' no good match found (score 52.94117647058824), logging.
⚠️ Logged 1 unmatched ingredients to 'unmatched_ingredients.csv'


## A helper function to bulk-process OCR outputs:



In [28]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

# Load your cleaned ingredient database
db = pd.read_csv("paulaschoice_us_ingredients_clean.csv")
db_names = db['Name'].tolist()

def clean_text(text):
    """Basic cleaning: lowercase, remove special chars."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def match_ingredient(ingredient, threshold=80):
    """Fuzzy match ingredient to database."""
    cleaned = clean_text(ingredient)
    result = process.extractOne(cleaned, db_names, scorer=fuzz.token_sort_ratio)
    if result:
        match, score = result[0], result[1]
    else:
        match, score = None, 0

    if score >= threshold:
        db_row = db[db['Name'] == match].iloc[0]
        return {
            "Input": ingredient,
            "Matched Name": db_row['Name'],
            "Description": db_row['Description'],
            "Rating": db_row['Rating'],
            "Score": score
        }
    else:
        return {
            "Input": ingredient,
            "Matched Name": None,
            "Description": None,
            "Rating": None,
            "Score": score
        }

def bulk_process(ingredient_list, threshold=80, unmatched_log="unmatched_ingredients.csv"):
    """Process list of ingredients, log unmatched."""
    results = []
    unmatched = []

    for ing in ingredient_list:
        res = match_ingredient(ing, threshold)
        results.append(res)
        if res['Matched Name'] is None:
            unmatched.append({"Input": ing, "Score": res['Score']})

    df_results = pd.DataFrame(results)
    if unmatched:
        df_unmatched = pd.DataFrame(unmatched)
        df_unmatched.to_csv(unmatched_log, index=False)
        print(f"⚠️ Logged {len(unmatched)} unmatched ingredients to '{unmatched_log}'")

    return df_results

# Example usage:

ocr_outputs = ["glycerine", "ethylhexyglycerin", "sodium hualuronate", "xyzrandomchemical"]
df_results = bulk_process(ocr_outputs)
print(df_results)






⚠️ Logged 1 unmatched ingredients to 'unmatched_ingredients.csv'
                Input        Matched Name  \
0           glycerine           Glycerine   
1   ethylhexyglycerin  Ethylhexylglycerin   
2  sodium hualuronate  Sodium Hyaluronate   
3   xyzrandomchemical                None   

                                         Description Rating      Score  
0  A naturally occurring substance in skin that h...   best  88.888889  
1  Skin-softening agent with hydration boosting p...   good  91.428571  
2  The more bio-available, salt form of hyaluroni...   best  83.333333  
3                                               None   None  52.941176  
